# BoardMind AI: Whiteboard Data Augmentation Experiments
**Notebook:** `06_whiteboard_augmentation_experiments.ipynb`  
**Author:** Senior Computer Vision & ML Research Engineer  
**Hardware:** Intel Core i9-13900H CPU, 16GB RAM, Windows  

### Objectives:
1. Quantify the severe geometric bias in the original whiteboard dataset (96.5% median area coverage).
2. Design controlled augmentation strategies to simulate real classroom deployment conditions:
   - **Geometric:** Perspective transformation, rotation, translation, scaling.
   - **Photometric:** HSV color jitter, brightness, contrast, random erasing (glare/reflection & occlusion simulation).
   - **Compositional:** Mosaic augmentation.
3. Compare augmentation strategies systematically against the baseline checkpoint.
4. Ensure bounding-box localization accuracy is preserved while improving out-of-domain generalization.

In [ ]:
import sys
import os
import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

DATA_YAML = PROJECT_ROOT / 'datasets' / 'whiteboard' / 'data.yaml'
REGISTRY_FILE = PROJECT_ROOT / 'runs' / 'whiteboard_optimization' / 'experiment_registry.json'

print(f"Project Root: {PROJECT_ROOT}")

## 1. Dataset Geometric Distribution & Bias Analysis
We inspect the bounding box distributions that justify the need for aggressive geometric augmentation.

In [ ]:
from audit_dataset import audit
# Run audit function to display split metrics
# audit()

## 2. Inspect Augmentation Variations in Experiment Registry
We compare the controlled augmentation experiments:
- **`exp1_aug_perspective_scale`**: Geometric perspective (0.001), rotation (10 deg), scale (0.7), translate (0.2)
- **`exp2_aug_lighting_glare`**: Elevated HSV jitter (0.6, 0.6) and random erasing (0.5)
- **`exp3_aug_combined_robust`**: Full combined geometric + photometric robustness

In [ ]:
if REGISTRY_FILE.exists():
    with open(REGISTRY_FILE, 'r') as f:
        registry = json.load(f)
    
    aug_rows = []
    for name, rec in registry.items():
        if rec.get('category') in ['data_augmentation', 'baseline']:
            m = rec['metrics_val']
            aug = rec['augmentations']
            aug_rows.append({
                'Experiment': name,
                'Description': rec['description'],
                'Perspective': aug.get('perspective', 0),
                'Degrees': aug.get('degrees', 0),
                'Scale': aug.get('scale', 0),
                'Erasing': aug.get('erasing', 0),
                'Precision': m['precision'],
                'Recall': m['recall'],
                'F1': m.get('f1_score', 0),
                'mAP50': m['mAP50'],
                'mAP50-95': m['mAP50_95']
            })
    
    df_aug = pd.DataFrame(aug_rows)
    display(df_aug)
else:
    print("Registry file not yet available.")

## 3. Visualize Augmented Training Batches
We inspect mosaic batches generated during training to verify realistic classroom perspective and scaling.

In [ ]:
runs_dir = PROJECT_ROOT / 'runs' / 'whiteboard_optimization'
batch_images = list(runs_dir.rglob('train_batch*.jpg'))

if batch_images:
    fig, axes = plt.subplots(1, min(2, len(batch_images)), figsize=(16, 8))
    if len(batch_images) == 1:
        axes = [axes]
    for ax, img_p in zip(axes, batch_images[:2]):
        im = cv2.imread(str(img_p))
        im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        ax.imshow(im_rgb)
        ax.set_title(img_p.parent.parent.name + ' - ' + img_p.name)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No train batch visualization images found yet.")